# 02 · Partición train/test

Separamos los datos limpios en entrenamiento y test **antes** del análisis exploratorio. Así el EDA y todas las decisiones que salgan de él se toman solo con el train, y el test queda intacto para la evaluación final (evitamos la fuga de información, o *data snooping*).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

semilla = 42
ruta = "../data/bronze/Property.csv" 

df = pd.read_csv(ruta, parse_dates=["date_added"])

columnas_esperadas = ["property_type", "price", "location", "location_id", "city",
                      "latitude", "longitude", "baths", "bedrooms", "date_added",
                      "Total_Area", "dist_centro_km"]
faltan = set(columnas_esperadas) - set(df.columns)
assert not faltan, f"Faltan columnas en los datos limpios: {faltan}"

df.shape

## El problema de los anuncios "gemelos"

La superficie y el precio son estándar (parcelas de 5, 7 o 10 marla, precios redondos) y los agentes publican la misma vivienda varias veces, a veces cambiando algún detalle. Muchos anuncios coinciden en ciudad, zona, tipo, superficie y precio, y pueden ser la misma vivienda. Si uno cae en train y su gemelo en test, el modelo habrá visto la respuesta antes de evaluarlo.

In [ ]:
CLAVE_GRUPO = ["city", "location", "property_type", "Total_Area", "price"]
grupo = df.groupby(CLAVE_GRUPO, sort=False).ngroup()

print(f"Anuncios: {len(df)} | grupos de posibles gemelos: {grupo.nunique()}")

# ¿Qué pasaría con una partición aleatoria normal?
idx_a, idx_b = train_test_split(df.index, test_size=0.2, random_state=semilla, stratify=df["city"])
gemelos = grupo.loc[idx_b].isin(grupo.loc[idx_a])
print(f"Partición aleatoria: anuncios del test con un gemelo en train: {gemelos.sum()} ({gemelos.mean():.1%})")

**Decisión.** Partición agrupada y estratificada con `StratifiedGroupKFold`:

- **Grupos:** ciudad, zona, tipo, superficie y precio. Todos los posibles gemelos van juntos a train o a test.
- **Estratificada por `city`:** las ciudades están desequilibradas (Faisalabad no llega al 4 %).
- **80/20:** de las 5 particiones, una es el test.
- **Semilla fija:** el resultado es reproducible.

Es una evaluación más exigente: mide si el modelo sabe tasar una vivienda que no ha visto, no si recuerda un anuncio repetido.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
idx_train, idx_test = next(sgkf.split(df, y=df["city"], groups=grupo))

train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)

print(f"Train: {len(train)} ({len(train) / len(df):.0%}) | Test: {len(test)} ({len(test) / len(df):.0%})")
print("Grupos compartidos:", len(set(grupo.iloc[idx_train]) & set(grupo.iloc[idx_test])))

In [ ]:
comparacion = pd.DataFrame({
    "train": train["city"].value_counts(normalize=True),
    "test": test["city"].value_counts(normalize=True),
}).round(3)
display(comparacion)

for nombre, datos in [("train", train), ("test", test)]:
    lp = np.log(datos["price"])
    print(f"log(price) {nombre}: media {lp.mean():.3f} | desv. {lp.std():.3f}")

**Resultado.** Train y test tienen la misma proporción de cada ciudad y la misma distribución del precio, y ningún posible gemelo está en los dos (frente al 52,2 % del test con una partición aleatoria).

Guardamos los dos conjuntos. **El EDA (`03_EDA`) trabaja solo con `train.csv`**; el test no se vuelve a mirar hasta evaluar el modelo final.

In [ ]:
Path("../data/silver").mkdir(parents=True, exist_ok=True)
train.to_csv("../data/silver/train.csv", index=False)
test.to_csv("../data/silver/test.csv", index=False)
print("Guardados train.csv y test.csv en data/silver/")